In [1]:
import sys
sys.path.append("..")     # pour que le notebook trouve le dossier src
import pandas as pd
import plotly.express as px
from src.scoring import charger_univers, recommander, CRITERES

pd.options.display.float_format = "{:,.2f}".format

In [2]:
univers = charger_univers()
profils = ["prudent", "equilibre", "dynamique"]
tops = {p: recommander(p, tableau=univers, nb_actions=10) for p in profils}

tops["prudent"][["nom", "secteur", "famille", "dernier_prix", "score"]]

,nom,secteur,famille,dernier_prix,score
ticker,,,,,
JNJ,Johnson & Johnson,Healthcare,Défensif,276.78,90.40
CS.PA,AXA SA,Financial Services,Défensif,42.60,89.90
KO,The Coca-Cola Company,Consumer Defensive,Défensif,94.51,89.50
ENGI.PA,Engie SA,Utilities,Défensif,23.43,88.80
BRK-B,Berkshire Hathaway Inc.,Financial Services,Défensif,545.20,81.30
ACA.PA,Crédit Agricole S.A.,Financial Services,Défensif,14.99,80.60
CVX,Chevron Corporation,Energy,Défensif,234.85,80.50
AI.PA,L'Air Liquide S.A.,Basic Materials,Défensif,180.90,79.80
XOM,ExxonMobil Holdings Corporation,Energy,Défensif,186.51,79.60


In [3]:
r = tops["prudent"]
points = [c for c in r.columns if c in CRITERES or c == "bonus_famille"]
long = r[points].reset_index().melt(id_vars="ticker", var_name="critère", value_name="points")

fig = px.bar(long, x="points", y="ticker", color="critère", orientation="h",
             title="Points de score par critère (profil prudent)")
fig.update_yaxes(autorange="reversed")
fig.show()

In [4]:
def carte(profil):
    t = univers.copy()
    t["groupe"] = "Autres actions"
    t.loc[tops[profil].index, "groupe"] = f"Recommandées ({profil})"
    return px.scatter(t, x="volatilite_annuelle", y="rendement_annuel",
                      color="groupe", hover_name="nom", title=f"Profil {profil}")

carte("prudent").show()
carte("dynamique").show()

In [5]:
colonnes = ["volatilite_annuelle", "rendement_annuel", "drawdown_max", "sharpe", "beta"]
comparaison = pd.DataFrame({p: univers.loc[tops[p].index, colonnes].mean() for p in profils})
comparaison["Tout l'univers"] = univers[colonnes].mean()
print("Actions en commun prudent/dynamique :", len(set(tops["prudent"].index) & set(tops["dynamique"].index)))
comparaison.T

Actions en commun prudent/dynamique : 4


,volatilite_annuelle,rendement_annuel,drawdown_max,sharpe,beta
prudent,0.19,0.20,-0.18,1.06,0.43
equilibre,0.21,0.27,-0.19,1.26,0.40
dynamique,0.29,0.35,-0.27,1.20,0.76
Tout l'univers,0.29,0.13,-0.35,0.51,0.89


In [6]:
univers.groupby("famille")[colonnes].mean().sort_values("volatilite_annuelle")

,volatilite_annuelle,rendement_annuel,drawdown_max,sharpe,beta
famille,,,,,
Défensif,0.22,0.14,-0.24,0.65,0.45
Modéré,0.30,0.25,-0.30,0.84,1.21
Dynamique,0.36,-0.11,-0.60,-0.33,1.01
Spéculatif,0.54,0.34,-0.54,0.65,2.08
